# Vanilla Transformer — Akkadian → English Machine Translation

**Architecture:** Standard encoder-decoder Transformer (Vaswani et al., 2017)

**Features:**
- ✅ **Multi-Head Self-Attention** (encoder & decoder) + **Cross-Attention** (decoder → encoder)
- ✅ **Positional Encoding** — sinusoidal, same as original paper
- ✅ **BPE Tokenisation** — same setup as BiLSTM baseline (src 4000, tgt 6000 vocab)
- ✅ **Label Smoothing** (ε=0.1) — same regularisation as BiLSTM baseline
- ✅ **Warmup + Inverse-sqrt LR Schedule** — standard Transformer schedule
- ✅ **Beam Search** decoding at inference
- ✅ Evaluation via BLEU + chrF++

## 0. Imports & Seeds

In [1]:
import sys
print(sys.executable)

c:\Users\surya\GPUenv\Scripts\python.exe


In [2]:
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sacrebleu import corpus_bleu, corpus_chrf
from tqdm import tqdm
import heapq

# BPE tokenizer — same library as BiLSTM baseline
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 1. Load & Clean Data

In [3]:
df = pd.read_csv("data.csv")

df = df[['transliteration', 'translation']].dropna()
df = df.rename(columns={'transliteration': 'source'})

def clean_text(text):
    text = text.lower()
    text = text.replace('"', '')
    text = text.replace('<gap>', ' <sep> ')
    return text

df['source']      = df['source'].apply(clean_text)
df['translation'] = df['translation'].apply(clean_text)

print(f"Total samples: {len(df)}")

Total samples: 62196


## 2. Train / Validation / Test Split

Split **before** BPE training to avoid data leakage. 80 / 10 / 10.

In [4]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 49756, Val: 6220, Test: 6220


## 3. Byte Pair Encoding (BPE) Tokenisation

Identical setup to the BiLSTM baseline — **separate** BPE models for Akkadian (4k vocab) and English (6k vocab),
trained only on the training split.

In [5]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<sos>", "<eos>", "<sep>"]

def train_bpe(texts, vocab_size):
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=SPECIAL_TOKENS,
        min_frequency=2
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)
    return tokenizer

print("Training source (Akkadian) BPE tokenizer...")
src_tokenizer = train_bpe(train_df['source'].tolist(), vocab_size=4000)

print("Training target (English) BPE tokenizer...")
tgt_tokenizer = train_bpe(train_df['translation'].tolist(), vocab_size=6000)

SRC_VOCAB_SIZE = src_tokenizer.get_vocab_size()
TGT_VOCAB_SIZE = tgt_tokenizer.get_vocab_size()
print(f"\nSource BPE vocab: {SRC_VOCAB_SIZE} | Target BPE vocab: {TGT_VOCAB_SIZE}")

Training source (Akkadian) BPE tokenizer...
Training target (English) BPE tokenizer...

Source BPE vocab: 4000 | Target BPE vocab: 6000


### 3a. Encode splits with BPE

In [6]:
def encode_bpe(texts, tokenizer):
    return [enc.ids for enc in tokenizer.encode_batch(texts)]

SRC_PAD_IDX = src_tokenizer.token_to_id("<pad>")
TGT_PAD_IDX = tgt_tokenizer.token_to_id("<pad>")
SOS_IDX     = tgt_tokenizer.token_to_id("<sos>")
EOS_IDX     = tgt_tokenizer.token_to_id("<eos>")

for split_df in [train_df, val_df, test_df]:
    split_df['src_ids'] = encode_bpe(split_df['source'].tolist(), src_tokenizer)
    tgt_ids_raw = encode_bpe(split_df['translation'].tolist(), tgt_tokenizer)
    split_df['tgt_ids'] = [[SOS_IDX] + ids + [EOS_IDX] for ids in tgt_ids_raw]

# Quick sanity check
sample = train_df.iloc[0]
print("Source  :", sample['source'])
print("BPE ids :", sample['src_ids'][:12], "...")
print("Decoded :", src_tokenizer.decode(sample['src_ids']))

Source  : á . s à g s a g .b a d a . nu n . n a . k ex . e . n e s a g . k i(var. .dul).bi h é .p à d : asakku ma- mÿt danunnakÿ ú-tam-me-ka - asakku-demon, i here- with exorcise you with the oath of the anunnaki
BPE ids : [118, 27, 68, 117, 56, 68, 50, 56, 27, 51, 50, 53] ...
Decoded : á . s à g s a g . b a d a . nu n . n a . k ex . e . n e s a g . k i ( var . . dul ). bi h é . p à d : a sak ku ma - mÿ t da nun nak ÿ ú - tam - me - ka - a sak ku - dem on , i her e - with ex or ci se you with the o ath of the a nun naki


### 3b. Filter by length

In [7]:
def filter_by_length(df, max_len=80):
    df['src_len'] = df['src_ids'].apply(len)
    df['tgt_len'] = df['tgt_ids'].apply(len)
    df = df[(df['src_len'] <= max_len) & (df['tgt_len'] <= max_len)]
    df = df.sort_values(by='src_len').reset_index(drop=True)
    return df

train_df = filter_by_length(train_df, 80)
val_df   = filter_by_length(val_df,   80)
test_df  = filter_by_length(test_df,  80)

print(f"After filtering → Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

After filtering → Train: 37325, Val: 4687, Test: 4653


## 4. Dataset & DataLoader

The Transformer expects **batch-first** tensors: `(batch, seq_len)`.

In [8]:
class TranslationDataset(Dataset):
    def __init__(self, df):
        self.src = df['src_ids'].tolist()
        self.tgt = df['tgt_ids'].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return torch.tensor(self.src[idx]), torch.tensor(self.tgt[idx])


def collate_fn(batch):
    """Pad to longest sequence in the batch; batch-first (batch, seq_len)."""
    src_seqs, tgt_seqs = zip(*batch)
    src_padded = pad_sequence(src_seqs, batch_first=True, padding_value=SRC_PAD_IDX)
    tgt_padded = pad_sequence(tgt_seqs, batch_first=True, padding_value=TGT_PAD_IDX)
    return src_padded, tgt_padded


BATCH_SIZE = 32

train_dataset = TranslationDataset(train_df)
val_dataset   = TranslationDataset(val_df)
test_dataset  = TranslationDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

Train batches: 1167 | Val: 147 | Test: 146


## 5. Model — Vanilla Transformer

### Architecture overview

```
SOURCE  →  Embedding + Positional Encoding
        →  [TransformerEncoderLayer × N]        (Multi-Head Self-Attention + FFN)
        →  memory  (batch, src_len, d_model)

TARGET  →  Embedding + Positional Encoding
        →  [TransformerDecoderLayer × N]        (Self-Attention + Cross-Attention + FFN)
        →  Linear projection → vocab logits
```

All components use **Pre-LN** (Layer Norm before sub-layer) via PyTorch's built-in `nn.Transformer` with `norm_first=True`,
which trains more stably than Post-LN on low-resource data.

In [9]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding (Vaswani et al., 2017).
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)                    # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)      # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )                                                      # (d_model/2,)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)                                   # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class TransformerMT(nn.Module):
    """
    Encoder-Decoder Transformer for sequence-to-sequence translation.

    Parameters
    ----------
    src_vocab_size : int
    tgt_vocab_size : int
    d_model        : int   — embedding / model dimension (default 256)
    nhead          : int   — number of attention heads  (default 8)
    num_encoder_layers : int — Transformer encoder depth (default 3)
    num_decoder_layers : int — Transformer decoder depth (default 3)
    dim_feedforward: int   — FFN hidden dim (default 512)
    dropout        : float — dropout probability        (default 0.1)
    max_len        : int   — max sequence length for positional encoding
    """
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model:             int   = 256,
        nhead:               int   = 8,
        num_encoder_layers:  int   = 3,
        num_decoder_layers:  int   = 3,
        dim_feedforward:     int   = 512,
        dropout:             float = 0.1,
        max_len:             int   = 512,
        src_pad_idx:         int   = 0,
        tgt_pad_idx:         int   = 0,
    ):
        super().__init__()

        self.d_model      = d_model
        self.src_pad_idx  = src_pad_idx
        self.tgt_pad_idx  = tgt_pad_idx

        # Embeddings + positional encoding
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=src_pad_idx)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=tgt_pad_idx)
        self.pos_encoder   = PositionalEncoding(d_model, dropout, max_len)
        self.pos_decoder   = PositionalEncoding(d_model, dropout, max_len)

        # Core Transformer
        # norm_first=True  →  Pre-LN  (more stable on small data)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_encoder_layers,
            norm=nn.LayerNorm(d_model)
        )
        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer, num_layers=num_decoder_layers,
            norm=nn.LayerNorm(d_model)
        )

        # Output projection
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

        self._init_weights()

    def _init_weights(self):
        """Xavier uniform for all linear / embedding weights."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    # ------------------------------------------------------------------
    # Mask helpers
    # ------------------------------------------------------------------

    def make_src_key_padding_mask(self, src: torch.Tensor) -> torch.Tensor:
        """
        src: (batch, src_len)
        Returns boolean mask (batch, src_len)  where True = PAD (ignored)
        """
        return src == self.src_pad_idx

    def make_tgt_key_padding_mask(self, tgt: torch.Tensor) -> torch.Tensor:
        """
        tgt: (batch, tgt_len)
        Returns boolean mask (batch, tgt_len)  where True = PAD (ignored)
        """
        return tgt == self.tgt_pad_idx

    @staticmethod
    def make_causal_mask(sz: int, device: torch.device) -> torch.Tensor:
        """
        Upper-triangular causal mask of shape (sz, sz).
        True values are masked out (future positions).
        """
        return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()

    # ------------------------------------------------------------------
    # Encode / decode sub-steps (useful at inference)
    # ------------------------------------------------------------------

    def encode(self, src: torch.Tensor) -> torch.Tensor:
        """
        src: (batch, src_len)
        Returns memory: (batch, src_len, d_model)
        """
        src_pad_mask = self.make_src_key_padding_mask(src)
        src_emb = self.pos_encoder(
            self.src_embedding(src) * math.sqrt(self.d_model)
        )
        return self.transformer_encoder(src_emb, src_key_padding_mask=src_pad_mask)

    def decode(self, tgt: torch.Tensor, memory: torch.Tensor,
               src: torch.Tensor) -> torch.Tensor:
        """
        tgt    : (batch, tgt_len)
        memory : (batch, src_len, d_model)
        src    : (batch, src_len)  — needed to rebuild src padding mask
        Returns logits: (batch, tgt_len, tgt_vocab)
        """
        tgt_len          = tgt.size(1)
        tgt_causal_mask  = self.make_causal_mask(tgt_len, tgt.device)
        tgt_pad_mask     = self.make_tgt_key_padding_mask(tgt)
        src_pad_mask     = self.make_src_key_padding_mask(src)

        tgt_emb = self.pos_decoder(
            self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        )
        out = self.transformer_decoder(
            tgt_emb, memory,
            tgt_mask=tgt_causal_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )
        return self.fc_out(out)    # (batch, tgt_len, tgt_vocab)

    # ------------------------------------------------------------------
    # Forward (training) — teacher-forced
    # ------------------------------------------------------------------

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        """
        src : (batch, src_len)   — source token ids
        tgt : (batch, tgt_len)   — target token ids  (includes <sos>, excludes final <eos> for input)
        Returns logits: (batch, tgt_len, tgt_vocab)

        During training we shift tgt:
          decoder input  = tgt[:, :-1]   (<sos> … last-token)
          target labels  = tgt[:,  1:]   (first-token … <eos>)
        """
        tgt_input = tgt[:, :-1]
        memory    = self.encode(src)
        logits    = self.decode(tgt_input, memory, src)
        return logits    # (batch, tgt_len-1, tgt_vocab)

## 6. Initialise Model, Optimiser & LR Schedule

**Warmup + Inverse-sqrt schedule** (Noam schedule, same as original paper):

$$lr = d_{model}^{-0.5} \cdot \min(step^{-0.5},\ step \cdot warmup\_steps^{-1.5})$$

This linearly warms up for `warmup_steps` then decays as `1/sqrt(step)`.

In [10]:
# ── Hyper-parameters ──────────────────────────────────────────────────
D_MODEL         = 256
NHEAD           = 8       # D_MODEL must be divisible by NHEAD
NUM_ENC_LAYERS  = 3
NUM_DEC_LAYERS  = 3
DIM_FEEDFORWARD = 512
DROPOUT         = 0.1
LABEL_SMOOTHING = 0.1
WARMUP_STEPS    = 4000

# ── Model ─────────────────────────────────────────────────────────────
model = TransformerMT(
    src_vocab_size    = SRC_VOCAB_SIZE,
    tgt_vocab_size    = TGT_VOCAB_SIZE,
    d_model           = D_MODEL,
    nhead             = NHEAD,
    num_encoder_layers= NUM_ENC_LAYERS,
    num_decoder_layers= NUM_DEC_LAYERS,
    dim_feedforward   = DIM_FEEDFORWARD,
    dropout           = DROPOUT,
    max_len           = 512,
    src_pad_idx       = SRC_PAD_IDX,
    tgt_pad_idx       = TGT_PAD_IDX,
).to(device)

# ── Optimiser ─────────────────────────────────────────────────────────
# Adam with β1=0.9, β2=0.98, ε=1e-9 as in the original Transformer paper
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

# ── Noam (warmup + inverse-sqrt) schedule ─────────────────────────────
def noam_lambda(step: int) -> float:
    step = max(step, 1)
    return (D_MODEL ** -0.5) * min(step ** -0.5, step * WARMUP_STEPS ** -1.5)

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=noam_lambda)

# ── Loss ──────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(
    ignore_index=TGT_PAD_IDX,
    label_smoothing=LABEL_SMOOTHING
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")
print(model)

c:\Users\surya\GPUenv\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Trainable parameters: 8,056,688
TransformerMT(
  (src_embedding): Embedding(4000, 256, padding_idx=0)
  (tgt_embedding): Embedding(6000, 256, padding_idx=0)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (pos_decoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dr

## 7. Inference Helpers

### 7a. Greedy decode

In [11]:
def translate_greedy(
    model, sentence, src_tokenizer, tgt_tokenizer, device, max_len=50
):
    """Greedy auto-regressive decode for a single sentence."""
    model.eval()
    src_ids    = src_tokenizer.encode(sentence).ids
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)   # (1, src_len)

    with torch.no_grad():
        memory = model.encode(src_tensor)                         # (1, src_len, d_model)

    generated = [SOS_IDX]

    for _ in range(max_len):
        tgt_tensor = torch.tensor(generated).unsqueeze(0).to(device)  # (1, t)
        with torch.no_grad():
            logits = model.decode(tgt_tensor, memory, src_tensor)      # (1, t, vocab)
        next_id = logits[0, -1].argmax().item()
        if next_id == EOS_IDX:
            break
        generated.append(next_id)

    return tgt_tokenizer.decode(generated[1:])   # strip <sos>

### 7b. Beam Search decode

Same beam search interface as the BiLSTM baseline — `beam_size` and `length_penalty` are configurable.

In [12]:
def translate_beam(
    model, sentence, src_tokenizer, tgt_tokenizer, device,
    beam_size=4, max_len=50, length_penalty=0.7
):
    """
    Beam search decoding for the Transformer.
    Maintains a set of (log_prob, token_ids) hypotheses and expands
    the top-beam_size candidates at each step.
    """
    model.eval()
    src_ids    = src_tokenizer.encode(sentence).ids
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)   # (1, src_len)

    with torch.no_grad():
        memory = model.encode(src_tensor)                         # (1, src_len, d_model)

    # Each hypothesis: (log_prob, token_ids_list)
    beams     = [(0.0, [SOS_IDX])]
    completed = []

    for _ in range(max_len):
        all_candidates = []

        for log_prob, ids in beams:
            tgt_tensor = torch.tensor(ids).unsqueeze(0).to(device)  # (1, t)
            with torch.no_grad():
                logits = model.decode(tgt_tensor, memory, src_tensor)  # (1, t, vocab)
            log_probs = F.log_softmax(logits[0, -1], dim=-1)           # (vocab,)
            topk_lp, topk_idx = log_probs.topk(beam_size)

            for lp, idx in zip(topk_lp.tolist(), topk_idx.tolist()):
                new_log_prob = log_prob + lp
                new_ids      = ids + [idx]
                if idx == EOS_IDX:
                    pen_score = new_log_prob / (len(ids) ** length_penalty)
                    completed.append((pen_score, ids[1:]))   # strip <sos>
                else:
                    all_candidates.append((new_log_prob, new_ids))

        # Prune to top beam_size
        beams = sorted(all_candidates, key=lambda x: x[0], reverse=True)[:beam_size]
        if not beams:
            break

    if completed:
        best_ids = max(completed, key=lambda x: x[0])[1]
    else:
        best_ids = beams[0][1][1:]  # strip <sos> from the best incomplete beam

    return tgt_tokenizer.decode(best_ids)

## 8. Evaluation Function

In [13]:
def evaluate_model(model, df_split, src_tokenizer, tgt_tokenizer, device,
                   beam=True, beam_size=4):
    """
    Evaluate on a dataset split.
    Returns BLEU, chrF++ (corpus-level).
    """
    preds, refs = [], []

    if beam:
        decode_fn = lambda s: translate_beam(
            model, s, src_tokenizer, tgt_tokenizer, device, beam_size=beam_size
        )
    else:
        decode_fn = lambda s: translate_greedy(
            model, s, src_tokenizer, tgt_tokenizer, device
        )

    for _, row in df_split.iterrows():
        src  = row['source']
        ref  = row['translation'].replace("<sep>", " ").strip()
        pred = decode_fn(src).replace("<sep>", " ").strip()
        preds.append(pred)
        refs.append(ref)

    bleu = corpus_bleu(preds, [refs]).score
    chrf = corpus_chrf(preds, [refs]).score
    return bleu, chrf

## 9. Training Loop

Key differences vs BiLSTM baseline:
- **No teacher forcing ratio** — Transformer is always fully teacher-forced during training (causal mask handles it)
- **Noam schedule** steps every batch, not every epoch
- **Gradient clipping** (`max_norm=1.0`) still applied
- Best checkpoint saved on `val_loss`; early stopping with `patience=10`

In [ ]:
N_EPOCHS  = 100
CLIP      = 1.0
PATIENCE  = 10
BEST_PATH = "transformer_akkadian_best.pt"

best_val_loss    = float('inf')
patience_counter = 0
history          = []
global_step      = 0   # for Noam scheduler

for epoch in range(N_EPOCHS):

    # ── Training ──────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS} [Train]"):
        src, tgt = src.to(device), tgt.to(device)
        # src: (batch, src_len)
        # tgt: (batch, tgt_len)  — includes both <sos> and <eos>

        optimizer.zero_grad()

        # model.forward shifts internally:
        #   decoder input  = tgt[:, :-1]   (from <sos> up to last token)
        #   logits shape   = (batch, tgt_len-1, vocab)
        logits = model(src, tgt)

        # Target labels: tgt[:, 1:]  (from first token up to <eos>)
        tgt_labels = tgt[:, 1:].reshape(-1)                     # (batch*(tgt_len-1),)
        logits_flat = logits.reshape(-1, TGT_VOCAB_SIZE)        # (batch*(tgt_len-1), vocab)

        loss = criterion(logits_flat, tgt_labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()

        # Noam schedule: step every batch
        global_step += 1
        scheduler.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ── Validation loss ───────────────────────────────────────────────
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            logits   = model(src, tgt)
            tgt_flat = tgt[:, 1:].reshape(-1)
            log_flat = logits.reshape(-1, TGT_VOCAB_SIZE)
            val_loss += criterion(log_flat, tgt_flat).item()

    avg_val_loss = val_loss / len(val_loader)

    # ── BLEU / chrF++ (greedy, fast) ──────────────────────────────────
    val_bleu, val_chrf = evaluate_model(
        model, val_df, src_tokenizer, tgt_tokenizer, device, beam=False
    )

    current_lr = optimizer.param_groups[0]['lr']

    history.append({
        'epoch':      epoch + 1,
        'train_loss': avg_train_loss,
        'val_loss':   avg_val_loss,
        'val_bleu':   val_bleu,
        'val_chrf':   val_chrf,
        'lr':         current_lr,
        'step':       global_step,
    })

    print(
        f"\nEpoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
        f"| Val BLEU: {val_bleu:.2f} | Val chrF++: {val_chrf:.2f} "
        f"| LR: {current_lr:.6f} | Step: {global_step}"
    )

    # ── Checkpoint / Early Stopping ───────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  ✅ New best model saved (val_loss={best_val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("🛑 Early stopping triggered")
            break

print("\nTraining complete.")

Epoch 1/100 [Train]: 100%|██████████| 1167/1167 [00:49<00:00, 23.51it/s]


## 10. Final Evaluation on Test Set

Load best checkpoint and evaluate with **beam search** (beam_size=4).

In [ ]:
model.load_state_dict(torch.load(BEST_PATH, map_location=device))

# Greedy
test_bleu_greedy, test_chrf_greedy = evaluate_model(
    model, test_df, src_tokenizer, tgt_tokenizer, device, beam=False
)

# Beam search
test_bleu_beam, test_chrf_beam = evaluate_model(
    model, test_df, src_tokenizer, tgt_tokenizer, device, beam=True, beam_size=4
)

print("=" * 52)
print("  TEST SET RESULTS  (Vanilla Transformer + BPE)")
print("=" * 52)
print(f"  Greedy  — BLEU: {test_bleu_greedy:.2f}  chrF++: {test_chrf_greedy:.2f}")
print(f"  Beam-4  — BLEU: {test_bleu_beam:.2f}  chrF++: {test_chrf_beam:.2f}")
print("=" * 52)

## 11. Training History

In [ ]:
history_df = pd.DataFrame(history)
print(history_df.to_string(index=False))

## 12. Attention Visualisation

Visualise the **cross-attention** weights from the last decoder layer — which source BPE tokens
each generated target token attends to (directly analogous to the Bahdanau attention heatmaps from the BiLSTM model).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


def get_cross_attention(
    model, sentence, src_tokenizer, tgt_tokenizer, device, max_len=50
):
    """
    Run greedy decode and collect cross-attention weights from the
    last decoder layer (averaged over all heads).

    Returns
    -------
    tgt_tokens   : list[str]
    src_tokens   : list[str]
    attn_matrix  : np.ndarray  shape (tgt_len, src_len)
    """
    model.eval()

    src_ids    = src_tokenizer.encode(sentence).ids
    src_tokens = [src_tokenizer.id_to_token(i) for i in src_ids]
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)

    # Register a forward hook on the last decoder layer's cross-attention
    cross_attn_weights = []

    def hook_fn(module, input, output):
        # nn.MultiheadAttention returns (attn_output, attn_weights)
        # attn_weights: (batch, nhead, tgt_len, src_len) when need_weights=True
        # PyTorch averages over heads by default when need_weights=True
        cross_attn_weights.append(output[1].detach().cpu())  # (1, tgt_t, src_len)

    # Hook the multihead_attn of the last decoder layer
    last_decoder_layer = model.transformer_decoder.layers[-1]
    hook = last_decoder_layer.multihead_attn.register_forward_hook(hook_fn)

    with torch.no_grad():
        memory = model.encode(src_tensor)

    generated = [SOS_IDX]
    attn_rows  = []

    for _ in range(max_len):
        cross_attn_weights.clear()
        tgt_t = torch.tensor(generated).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model.decode(tgt_t, memory, src_tensor)
        # cross_attn_weights[0]: (1, t, src_len) — attention for all positions
        # We only want the last generated position
        attn_last = cross_attn_weights[0][0, -1, :len(src_ids)].numpy()  # (src_len,)
        next_id   = logits[0, -1].argmax().item()
        if next_id == EOS_IDX:
            break
        generated.append(next_id)
        attn_rows.append(attn_last)

    hook.remove()

    tgt_tokens  = [tgt_tokenizer.id_to_token(i) for i in generated[1:]]
    attn_matrix = np.array(attn_rows)   # (tgt_len, src_len)
    return tgt_tokens, src_tokens, attn_matrix


def plot_attention(sentence, tgt_tokens, src_tokens, attn_matrix):
    fig, ax = plt.subplots(
        figsize=(max(8, len(src_tokens)), max(6, len(tgt_tokens) // 2))
    )
    im = ax.matshow(attn_matrix, cmap='viridis')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(src_tokens)))
    ax.set_yticks(range(len(tgt_tokens)))
    ax.set_xticklabels(src_tokens, rotation=90, fontsize=9)
    ax.set_yticklabels(tgt_tokens, fontsize=9)
    ax.set_xlabel('Source (Akkadian BPE tokens)')
    ax.set_ylabel('Target (English BPE tokens)')
    ax.set_title(f'Cross-Attention (last decoder layer): "{sentence[:60]}"')
    plt.tight_layout()
    plt.savefig('transformer_attention_map.png', dpi=150)
    plt.show()


# Run on a sample from the test set
sample_sentence = test_df.iloc[0]['source']
tgt_tok, src_tok, attn = get_cross_attention(
    model, sample_sentence, src_tokenizer, tgt_tokenizer, device
)
plot_attention(sample_sentence, tgt_tok, src_tok, attn)

## 13. Sample Translations

In [ ]:
print("=" * 70)
print("  SAMPLE TRANSLATIONS (Beam-4)")
print("=" * 70)

for i in range(min(5, len(test_df))):
    row  = test_df.iloc[i]
    src  = row['source']
    ref  = row['translation'].replace("<sep>", " ").strip()
    pred = translate_beam(
        model, src, src_tokenizer, tgt_tokenizer, device, beam_size=4
    ).replace("<sep>", " ").strip()

    print(f"\n[{i+1}]")
    print(f"  SRC  : {src}")
    print(f"  REF  : {ref}")
    print(f"  PRED : {pred}")
print("=" * 70)